# 📂 Phase 2: Insurance Variables — Data Loading

This notebook loads and merges insurance-related variables for California counties from various sources, including the California Department of Insurance (CDI), FEMA, NAIC, and the U.S. Census API.  
The goal is to enrich the extended county-level GeoDataFrame (`gdf_ca_extended`) with coverage, claims, and other indicators for earthquake insurance risk analysis.

🔹 This notebook is dedicated to **data loading only**.  
🔹 **Data cleaning**, **transformations**, and **exploratory analysis (EDA)** will be performed in a separate notebook.

---

📑 **Notebook Contents**
- 🛡️ Earthquake Insurance Take-Up Rate
- 🧰 Insurance Claims & Loss Ratios
- 🏠 Homeowners Insurance Coverage
- 🆘 FEMA Disaster Assistance
- 📎 Appendix: Statewide Indicators

---

## 🛡️ Planned Insurance Variables

| Variable Group                        | Source                            | Description |
|---------------------------------------|-----------------------------------|-------------|
| **Earthquake Insurance Take-Up Rate** | California Department of Insurance (CDI) | % of homes with active earthquake insurance |
| **Insurance Claims & Loss Ratios**    | CDI or NAIC Reports               | Number of claims filed and payouts post-earthquake |
| **Homeowners Insurance Coverage**     | ACS / CDI                         | General homeowners insurance coverage rates |
| **FEMA Disaster Assistance**          | FEMA Disaster Declarations        | Public assistance payouts related to earthquakes |
| **Underinsured or At-Risk Regions** (*EDA phase*) | *Derived Variable*               | Counties with high seismic risk but low insurance penetration; will be created during EDA |

✅ The first four variables are loaded and merged in this notebook.  
🛠️ The final derived variable (**Underinsured or At-Risk Regions**) will be constructed in the next notebook, after cleaning and analysis.

---

## 📦 Load Packages


In [3]:
import pandas as pd
import geopandas as gpd
import requests


---
## Load California counties GeoJSON created in Phase 2
This file was saved at the end of `02_demographics_and_housing_data.ipynb`


In [5]:
# Load Extended County GeoDataFrame (Demographics + Housing)
gdf_ca_extended = gpd.read_file("../output/gdf_ca_extended_v1.geojson")
print(f"Loaded extended GeoDataFrame with {len(gdf_ca_extended)} counties")

gdf_ca_extended.head()


Loaded extended GeoDataFrame with 58 counties


,STATEFP,COUNTYFP,COUNTYNS,GEOID,GEOIDFQ,county,NAMELSAD,LSAD,CLASSFP,MTFCC,...,renter_pct,for_rent_pct,rented_not_occupied_pct,for_sale_pct,sold_not_occupied_pct,seasonal_pct,migrant_pct,other_vacant_pct,median_home_value,geometry
0,06,091,00277310,06091,0500000US06091,Sierra,Sierra County,06,H1,G4020,...,21.724710,0.000000,0.000000,4.143337,1.231803,74.132139,0.000000,20.492721,334100,"POLYGON ((-120.55587 39.50874, -120.55614 39.5..."
1,06,067,00277298,06067,0500000US06067,Sacramento,Sacramento County,06,H1,G4020,...,41.371081,34.907258,5.403064,8.187296,6.868034,9.226952,0.074824,35.332572,498900,"POLYGON ((-121.43991 38.25553, -121.44002 38.2..."
2,06,083,00277306,06083,0500000US06083,Santa Barbara,Santa Barbara County,06,H1,G4020,...,47.130773,20.636683,7.255229,5.996669,6.570424,34.101425,0.814362,24.625208,735700,"MULTIPOLYGON (((-120.58226 34.10752, -120.5790..."
3,06,009,01675885,06009,0500000US06009,Calaveras,Calaveras County,06,H1,G4020,...,15.633905,1.766164,0.444123,2.891964,2.726709,82.142119,0.206569,9.822351,441800,"POLYGON ((-120.6318 38.34603, -120.63066 38.34..."
4,06,111,00277320,06111,0500000US06111,Ventura,Ventura County,06,H1,G4020,...,35.842040,23.750452,2.890521,5.654583,4.251475,34.716368,0.397447,28.339155,768400,"MULTIPOLYGON (((-119.63607 33.28071, -119.6348..."


---
## 🛡️ Earthquake Insurance Take-Up Rate

**Source:** California Department of Insurance (CDI)  
**Report:** *The Number of Residential Properties with Fire or Earthquake Coverage in Calendar Year 2023*

**Metric:** Share of residential **households** with active earthquake insurance policies.  
Used to assess market penetration and identify underinsured, high-risk regions.

### 📊 Variables Included
- `eq_policies`: Total residential earthquake policies (across all policy types)
- `total_households`: Total occupied housing units per county (ACS 2023)
- `eq_takeup_pct`: Earthquake insurance take-up rate (policies ÷ households)

### 📐 Calculation Formula

$$
\
\text{eq\_takeup\_pct} = \frac{\text{eq\_policies}}{\text{total\_households}}
\
$$

> **Note:** This notebook uses the total number of residential earthquake policies as reported by CDI, divided by the number of **occupied** housing units (households), for consistency with insurance eligibility.

### 🏠 Policy Types Included (per CDI)
- **Homeowners**: HO-2, HO-3, HO-5, HO-8 (includes dwelling fire for owners)
- **Dwelling Fire – Tenant Occupied**: Common for rentals
- **Condominium (HO-6)**
- **Mobile Home**
- **Renters (HO-4)**

Each policy type includes:
- Fire policy count
- Earthquake policy count
- % with earthquake coverage

📘 These categories represent the full residential market tracked in the CDI summary report.


In [7]:
# Load CDI Earthquake Policy Counts (from Excel export)
path = "../data/CDI_2023_Fire_EQ_Policy_Counts_by_County.xlsx"
df_cdi = pd.read_excel(path)

# Preview shape and structure
print(f"Loaded Excel with shape: {df_cdi.shape}")
df_cdi.head()


Loaded Excel with shape: (58, 16)


,County,HO_Fire,HO_EQ,HO_Pct,DF_Fire,DF_EQ,DF_Pct,Condo_Fire,Condo_EQ,Condo_Pct,Mobile_Fire,Mobile_EQ,Mobile_Pct,Rent_Fire,Rent_EQ,Rent_Pct
0,ALAMEDA,271034,32157,0.12,62720,3687,0.06,56066,3238,0.06,3895,626,0.16,141315,16064,0.11
1,ALPINE,729,74,0.10,64,3,0.05,313,15,0.05,7,-,0.00,29,4,0.15
2,AMADOR,14721,714,0.05,1836,57,0.03,68,3,0.04,1278,132,0.10,1572,159,0.10
3,BUTTE,42021,2405,0.06,10099,272,0.03,993,43,0.04,7097,821,0.12,16377,1604,0.10
4,CALAVERAS,24430,862,0.04,2420,60,0.02,399,20,0.05,1608,137,0.09,1487,161,0.11


In [8]:
# Count unique counties to verify completeness
print("Number of counties:", df_cdi["County"].nunique())


Number of counties: 58


In [9]:
# Convert EQ policy columns to numeric
eq_columns = ["HO_EQ", "DF_EQ", "Condo_EQ", "Mobile_EQ", "Rent_EQ"]
df_cdi[eq_columns] = df_cdi[eq_columns].apply(pd.to_numeric, errors="coerce")

# Replace missing values and convert to integer
df_cdi[eq_columns] = df_cdi[eq_columns].fillna(0).astype(int)

# Clean county name formatting
df_cdi["County"] = df_cdi["County"].str.title().str.strip()

# Sum all EQ policy types into total
df_cdi["eq_policies"] = df_cdi[eq_columns].sum(axis=1)

# Keep relevant columns only
df_eq = df_cdi[["County", "eq_policies"]].copy()
df_eq.columns = ["county", "eq_policies"]  

# Preview: Top counties by EQ policy count
df_eq.sort_values("eq_policies", ascending=False).head(10)


,county,eq_policies
18,Los Angeles,458782
36,San Diego,269587
29,Orange,175944
32,Riverside,78958
55,Ventura,60466
42,Santa Clara,57841
0,Alameda,55772
35,San Bernardino,50989
37,San Francisco,41628
14,Kern,34966


In [10]:
# Merge EQ policy counts into gdf
gdf_ca_extended = gdf_ca_extended.merge(df_eq, how="left", on="county")

# Calculate Earthquake Insurance Take-Up Rate (%)
gdf_ca_extended["eq_takeup_pct"] = (
    gdf_ca_extended["eq_policies"] / gdf_ca_extended["total_households"]
) * 100


✅ Earthquake insurance policy counts and take-up rate successfully merged into `gdf_ca_extended`.  
Includes total residential policies and take-up percentage based on occupied housing units.

Use: `gdf_ca_extended[['county', 'eq_policies', 'total_households', 'eq_takeup_pct']]` to view or validate.


In [12]:
# ✅ Preview: Top 10 counties by take-up rate
gdf_ca_extended[['county', 'eq_policies', 'total_households', 'eq_takeup_pct']]\
    .sort_values('eq_takeup_pct', ascending=False)\
    .head(10)


,county,eq_policies,total_households,eq_takeup_pct
20,Mono,2886,5521,52.273139
57,San Luis Obispo,28672,108897,26.329467
8,San Diego,269587,1159822,23.243825
19,Inyo,1831,7923,23.109933
4,Ventura,60466,278045,21.746840
25,Alpine,96,473,20.295983
11,Marin,17886,103201,17.331227
47,Orange,175944,1074654,16.372153
2,Santa Barbara,23993,148960,16.107009
55,Santa Cruz,15036,96873,15.521353


---
## 🧰 Insurance Claims & Loss Ratios


### 📉 Modeled Loss Ratio (CDI PML Report)

**Source**: California Department of Insurance (CDI)  
*2011–2018 CA Earthquake Probable Maximum Loss Report*   
Form A, Appendix I, 2018

The **Modeled Loss Ratio** estimates the severity of potential insured earthquake losses relative to total insured property liability. It reflects the **expected losses from a maximum-size earthquake** likely to occur in each region, based on a **250-year return period** earthquake scenario commonly used in catastrophe risk modeling.

### 📐 Calculation Formula

$$
\
\text{Modeled Loss Ratio} = \frac{\text{Direct Insured PML}_{2018}}{\text{Total Direct Liability}_{2018}}
\
$$

Where:  
- **Direct Insured PML**: Expected monetary loss after deductibles, in millions of dollars
- **Total Direct Liability**: Total insured property value at risk, in millions of dollars  

**Notes**:
- The report provides **zone-level estimates** (Zones A–H), which were joined to California counties using the official **Earthquake Zones by County** mapping from the same report.
- PML values represent modeled losses from **vibratory ground motion only**, and actual losses may vary depending on soil conditions, structural types, and proximity to fault lines.
- These estimates are based on insurer-reported data for earthquake insurance policies in effect as of **December 31, 2018**.


In [14]:
# Load Earthquake Zone Mapping and Loss Ratio Data
zones_by_county = pd.read_excel("../data/earthquake_zones_by_county.xlsx")
zone_totals = pd.read_excel("../data/earthquake_zone_totals_2018.xlsx")

# Merge zone-level risk estimates
gdf_ca_extended = gdf_ca_extended.merge(zones_by_county, on="county", how="left")
gdf_ca_extended = gdf_ca_extended.merge(zone_totals, on="EQ_Zone", how="left")

gdf_ca_extended.head()


,STATEFP,COUNTYFP,COUNTYNS,GEOID,GEOIDFQ,county,NAMELSAD,LSAD,CLASSFP,MTFCC,...,migrant_pct,other_vacant_pct,median_home_value,geometry,eq_policies,eq_takeup_pct,EQ_Zone,Region,Total_Direct_Liability_2018,Direct_Insured_PML_2018
0,06,091,00277310,06091,0500000US06091,Sierra,Sierra County,06,H1,G4020,...,0.000000,20.492721,334100,"POLYGON ((-120.55587 39.50874, -120.55614 39.5...",107,8.872305,H,North,2900.0,203.0
1,06,067,00277298,06067,0500000US06067,Sacramento,Sacramento County,06,H1,G4020,...,0.074824,35.332572,498900,"POLYGON ((-121.43991 38.25553, -121.44002 38.2...",31509,5.545182,G,North-Central,56727.0,6631.0
2,06,083,00277306,06083,0500000US06083,Santa Barbara,Santa Barbara County,06,H1,G4020,...,0.814362,24.625208,735700,"MULTIPOLYGON (((-120.58226 34.10752, -120.5790...",23993,16.107009,C,Santa Barbara,93527.0,7076.0
3,06,009,01675885,06009,0500000US06009,Calaveras,Calaveras County,06,H1,G4020,...,0.206569,9.822351,441800,"POLYGON ((-120.6318 38.34603, -120.63066 38.34...",1240,6.928536,G,North-Central,56727.0,6631.0
4,06,111,00277320,06111,0500000US06111,Ventura,Ventura County,06,H1,G4020,...,0.397447,28.339155,768400,"MULTIPOLYGON (((-119.63607 33.28071, -119.6348...",60466,21.746840,C,Santa Barbara,93527.0,7076.0


In [15]:
# Sample validation
gdf_ca_extended[["county", "EQ_Zone", "Total_Direct_Liability_2018", "Direct_Insured_PML_2018"]].sample(5)


,county,EQ_Zone,Total_Direct_Liability_2018,Direct_Insured_PML_2018
44,Merced,F,12035.0,2389.0
21,Tuolumne,NaN,NaN,NaN
52,Plumas,H,2900.0,203.0
7,Kings,F,12035.0,2389.0
25,Alpine,E,90122.0,13382.0


In [16]:
# Ensure financial columns are numeric
gdf_ca_extended["Total_Direct_Liability_2018"] = pd.to_numeric(
    gdf_ca_extended["Total_Direct_Liability_2018"], errors="coerce"
)
gdf_ca_extended["Direct_Insured_PML_2018"] = pd.to_numeric(
    gdf_ca_extended["Direct_Insured_PML_2018"], errors="coerce"
)

# Calculate Modeled Loss Ratio (%)
gdf_ca_extended["Modeled_Loss_Ratio_2018_Pct"] = (
    gdf_ca_extended["Direct_Insured_PML_2018"] / gdf_ca_extended["Total_Direct_Liability_2018"]
) * 100


✅ Insurance Claims & Loss Ratios successfully added into gdf_ca_extended.  
Use:
`gdf_ca_extended[["county", "EQ_Zone", "Total_Direct_Liability_2018", "Direct_Insured_PML_2018", "Modeled_Loss_Ratio_2018_Pct"]]` to view or validate.


In [18]:
# ✅ Preview: Highest modeled loss ratios
gdf_ca_extended[[
    "county",
    "EQ_Zone",
    "Total_Direct_Liability_2018",
    "Direct_Insured_PML_2018",
    "Modeled_Loss_Ratio_2018_Pct"
]].sort_values("Modeled_Loss_Ratio_2018_Pct", ascending=False).head(10)


,county,EQ_Zone,Total_Direct_Liability_2018,Direct_Insured_PML_2018,Modeled_Loss_Ratio_2018_Pct
45,Tulare,F,12035.0,2389.0,19.850436
12,Mariposa,F,12035.0,2389.0,19.850436
44,Merced,F,12035.0,2389.0,19.850436
37,Madera,F,12035.0,2389.0,19.850436
36,Fresno,F,12035.0,2389.0,19.850436
7,Kings,F,12035.0,2389.0,19.850436
48,Imperial,E,90122.0,13382.0,14.848761
31,Riverside,E,90122.0,13382.0,14.848761
25,Alpine,E,90122.0,13382.0,14.848761
20,Mono,E,90122.0,13382.0,14.848761


---
## 🏠 Homeowners Insurance Coverage

**Source:** California Department of Insurance (CDI)  
**Report:** *Number of New, Renewed, and Non-Renewed Homeowners' Policies by County (2023)*

**Metric:** Estimated share of residential properties with active homeowners insurance coverage.

This variable reflects actual **policy counts** in the voluntary and FAIR Plan markets, merged with housing unit totals from the U.S. Census (ACS 2023) to estimate coverage rates.

### 📊 Variables Included
- `Total_Homeowners_Policies`: Sum of new + renewed policies (voluntary + FAIR)
- `total_housing_units`: Total residential units (includes vacant)
- `homeowners_coverage_pct`: Ratio of policies to housing units (proxy for take-up)

### ✅ Coverage Types Included
- Homeowners forms: **HO-2**, **HO-3**, **HO-5**, **HO-8**
- **Dwelling fire** (structure only, not contents-only)
- **Landlord policies** (≤4-unit residential)
- **Mobile home** insurance

📌 Represents ~99% of the homeowners market.  
❌ **Excludes:** HO-4 (renters) and HO-6 (condo unit-owners).

### 🛡️ Market Breakdown
- **Voluntary Market**: Standard private carriers
- **FAIR Plan**: State-backed insurer of last resort for high-risk zones


In [20]:
# Load Homeowners Policy Counts by County
path = "../data/CDI_2023_Homeowners_Policy_Counts_by_County.xlsx"
df_homeowners = pd.read_excel(path)

df_homeowners.head()


,county,New_Voluntary,Renewed_Voluntary,New_FAIR,Renewed_FAIR
0,ALAMEDA,25158.0,318139.0,1021.0,2359.0
1,ALPINE,17.0,443.0,68.0,163.0
2,AMADOR,706.0,9580.0,1101.0,3519.0
3,BUTTE,4836.0,46091.0,3046.0,3168.0
4,CALAVERAS,792.0,14025.0,2081.0,6270.0


In [21]:
# Clean county names
df_homeowners["county"] = df_homeowners["county"].str.title().str.strip()

# Total policies = new + renewed (voluntary + FAIR)
df_homeowners["Total_Homeowners_Policies"] = (
    df_homeowners["New_Voluntary"]
    + df_homeowners["Renewed_Voluntary"]
    + df_homeowners["New_FAIR"]
    + df_homeowners["Renewed_FAIR"]
)

# Confirm total policies across all counties
df_homeowners["Total_Homeowners_Policies"].sum()


8623350.0

In [22]:
# Merge homeowners data into GeoDataFrame
gdf_ca_extended = gdf_ca_extended.merge(
    df_homeowners,
    how="left",
    on="county",
)

# Calculate homeowners insurance coverage rate (%)
gdf_ca_extended["homeowners_coverage_pct"] = (
    gdf_ca_extended["Total_Homeowners_Policies"] / gdf_ca_extended["total_housing_units"]
) * 100


✅ Homeowners insurance variables successfully merged into `gdf_ca_extended`.  
Includes total policy counts and derived coverage rates.  
Use: `gdf_ca_extended[['county', 'Total_Homeowners_Policies', 'total_housing_units', 'homeowners_coverage_pct']]` to view or validate.


In [24]:
# ✅ Preview: Counties with highest coverage rates
gdf_ca_extended[[
    "county", "Total_Homeowners_Policies", "total_housing_units", "homeowners_coverage_pct"
]].sort_values("homeowners_coverage_pct", ascending=False).head(10)


,county,Total_Homeowners_Policies,total_housing_units,homeowners_coverage_pct
3,Calaveras,23168.0,27579,84.005947
13,Lassen,10235.0,12232,83.673970
29,San Benito,17451.0,20937,83.350050
42,Nevada,42811.0,53981,79.307534
50,Amador,14906.0,18919,78.788519
37,Madera,39429.0,50368,78.281846
9,Placer,136658.0,176233,77.543933
21,Tuolumne,24071.0,31491,76.437712
26,El Dorado,71606.0,94258,75.968088
15,Shasta,60056.0,79748,75.307218


---
## 🆘 FEMA Disaster Assistance


Source: [FEMA Public Assistance API v2](https://www.fema.gov/openfema-data-page/public-assistance-funded-projects-details-v2)

This variable represents the **total federal funding obligated** to each California county under FEMA’s Public Assistance Program for **earthquake-related disaster declarations**.

- Variable: `fema_funding_earthquake`
- Units: USD (Federal Share Obligated)
- Time range: All earthquakes declared in CA (historical to present)
- Purpose: Used to identify **counties with high historical earthquake losses**, especially when combined with insurance coverage and seismic risk.


In [26]:
# FEMA Public Assistance API — Earthquake Disasters
url = "https://www.fema.gov/api/open/v2/PublicAssistanceFundedProjectsDetails"

# API parameters
params = {
    "$filter": "stateAbbreviation eq 'CA' and incidentType eq 'Earthquake'",
    "$format": "json",
    "$top": 1000
}

# Send request
response = requests.get(url, params=params)
data = response.json()

# Load into DataFrame
df_fema = pd.DataFrame(data['PublicAssistanceFundedProjectsDetails'])

df_fema.head()


,disasterNumber,declarationDate,incidentType,pwNumber,applicationTitle,applicantId,damageCategoryCode,damageCategoryDescrip,projectStatus,projectProcessStep,...,projectAmount,federalShareObligated,totalObligated,lastObligationDate,firstObligationDate,mitigationAmount,gmProjectId,gmApplicantId,lastRefresh,hash
0,1505,2004-01-13T00:00:00.000Z,Earthquake,9,(PW# 9) ROAD REPAIRS,079-49362-00,C,Roads and Bridges,Active,Project Closed Out,...,18094.00,13570.50,14536.72,2004-02-20T00:00:00.000Z,2004-02-20T00:00:00.000Z,0.0,412211,122721,2025-07-22T15:05:39.468Z,90bf22891d8015455fc82e673998cdc440aa6ed2
1,1505,2004-01-13T00:00:00.000Z,Earthquake,10,(PW# 10) SEWAGE PUMPING STATION REPAIR,079-56056-00,F,Utilities,Active,Project Closed Out,...,4700.00,3525.00,3775.98,2004-02-23T00:00:00.000Z,2004-02-23T00:00:00.000Z,0.0,412224,122722,2025-07-22T15:05:39.468Z,59da52a5261c542ad248e5bb23e4939967e39302
2,1505,2004-01-13T00:00:00.000Z,Earthquake,11,(PW# 11) BUILDING REPAIR - FIRE STATION,079-56056-00,E,Buildings and Equipment,Active,Project Closed Out,...,3624.00,2718.00,2911.52,2004-02-23T00:00:00.000Z,2004-02-23T00:00:00.000Z,0.0,412225,122722,2025-07-22T15:05:39.468Z,613e965a417596a30dc8fb997bde5cba6c60e8e3
3,1505,2004-01-13T00:00:00.000Z,Earthquake,12,(PW# 12) BUILDING REPAIR - TRANSPORTATION CENTER,079-56056-00,E,Buildings and Equipment,Active,Project Closed Out,...,1454.40,1090.80,1168.46,2004-02-23T00:00:00.000Z,2004-02-23T00:00:00.000Z,0.0,412226,122722,2025-07-22T15:05:39.468Z,67efef6b1809eb4bf5045a48d4a856159dbb567e
4,1505,2004-01-13T00:00:00.000Z,Earthquake,13,(PW# 13) WASTE WATER TREATMENT PLANT REPAIR,079-56056-00,F,Utilities,Active,Project Closed Out,...,7728.49,5796.37,6209.08,2004-02-23T00:00:00.000Z,2004-02-23T00:00:00.000Z,0.0,412227,122722,2025-07-22T15:05:39.468Z,8a52191c849df84aef2769bce2ce113e14eea2d5


In [27]:
# Check dataset size to see if pagination is required
print(f"shape {df_fema.shape}")

# Preview county names before cleanup
df_fema['county'].unique()


shape (660, 25)


array(['San Luis Obispo County', 'Santa Barbara County',
       'Sacramento County', 'Statewide', 'Imperial County', 'Napa County',
       'Solano County', 'Sierra County', 'Marin County'], dtype=object)

In [28]:
# Clean FEMA county names
df_fema = df_fema[df_fema['county'] != 'Statewide']  # Drop "Statewide"
df_fema['county'] = df_fema['county'].str.replace(" County", "", regex=False)
df_fema['county'] = df_fema['county'].str.strip().str.title()

# Aggregate FEMA funding by county
fema_by_county = (
    df_fema.groupby('county')['federalShareObligated']
    .sum()
    .reset_index()
    .rename(columns={'federalShareObligated': 'fema_funding_earthquake'})
)

fema_by_county.head(10)


,county,fema_funding_earthquake
0,Imperial,25023516.25
1,Marin,8311.40
2,Napa,21048662.00
3,Sacramento,44911.70
4,San Luis Obispo,44114267.10
5,Santa Barbara,648440.19
6,Sierra,227958.95
7,Solano,3096552.55


In [29]:
# Merge FEMA funding into gdf
gdf_ca_extended = gdf_ca_extended.merge(
    fema_by_county,
    on='county',
    how='left'
)


✅ FEMA Earthquake Assistance Successfully Merged into `gdf_ca_extended`.  
Use: `gdf_ca_extended[['county', 'fema_funding_earthquake']]` to view or validate.


In [31]:
# ✅ Preview: Counties with highest FEMA EQ funding
gdf_ca_extended[['county', 'fema_funding_earthquake']]\
    .sort_values('fema_funding_earthquake', ascending=False)\
    .head(10)


,county,fema_funding_earthquake
57,San Luis Obispo,44114267.10
48,Imperial,25023516.25
14,Napa,21048662.00
22,Solano,3096552.55
2,Santa Barbara,648440.19
0,Sierra,227958.95
1,Sacramento,44911.70
11,Marin,8311.40
3,Calaveras,NaN
4,Ventura,NaN


---
### 🔜 Underinsured or At-Risk Regions (Derived Variable)

This variable will be created in the next notebook during data cleaning and EDA.  
It will combine multiple indicators:

- Earthquake risk level  
- Earthquake insurance take-up rate  
- FEMA disaster assistance  
- Socioeconomic vulnerability (e.g., poverty, housing age, race/ethnicity)

**Purpose:** Identify counties at **high risk of financial loss** due to low insurance coverage and high hazard exposure.  
This will be used to flag potentially **underinsured regions** and support risk mitigation efforts.

---


In [33]:
# Save final extended GeoDataFrame (v2 includes insurance variables)
gdf_ca_extended.to_file("../output/gdf_ca_extended_v2.geojson", driver="GeoJSON")
gdf_ca_extended.to_csv("../output/gdf_ca_extended_v2.csv", index=False)

print("✅ gdf_ca_extended saved as both GeoJSON and CSV in /output/")


✅ gdf_ca_extended saved as both GeoJSON and CSV in /output/


### ✅ Final Note

All insurance-related variables have been successfully loaded and merged into `gdf_ca_extended`.  
This notebook is dedicated to **data loading only**, and does not perform cleaning, transformations, or encoding.

➡️ Data cleaning, imputation, one-hot encoding, and feature engineering (e.g., `fema_received`) will be performed in the **EDA notebook**.


---
## 📎 Appendix: Statewide Earthquake Insurance Indicators
The following section contains supplemental statewide metrics from CDI reports. These are not merged into `gdf_ca_extended`, but support analysis and interpretation.

### 📊 Statewide EQ Policy Count Cross-Check

Our cleaned CDI county-level dataset reports a total of:

> 🧮 **1,640,957** residential earthquake insurance policies in California (2023)

This figure is calculated by summing policy counts across all 58 counties and all residential policy types:

- Homeowners (HO)
- Dwelling Fire (tenant-occupied)
- Condo
- Mobile Home
- Renters

📄 The official CDI summary report  
(*2023 Earthquake Premium & Policy Count Data Call – Residential Totals*) reports:

> 🧾 **1,615,561** residential earthquake insurance policies statewide (2023)

**Difference:**  
1,640,957 − 1,615,561 = **25,396 policies** (approx. **1.6%** higher)

> ✅ This small discrepancy is expected and acceptable.  
> It likely reflects differences in policy definitions, timing, or inclusion of additional forms (e.g., small carriers, landlord policies) in the county-level dataset.


In [36]:
# Earthquake Insurance — Statewide Contextual Indicators

# Average Premium (CDI 2023)
total_eq_premiums_2023 = 1_472_288_223
total_eq_policies_2023 = 1_615_561
avg_eq_premium = total_eq_premiums_2023 / total_eq_policies_2023
print(f"Average Premium per Policy (2023): ${avg_eq_premium:,.2f}")

# Earthquake Insurance Loss Ratio (2023)
earthquake_loss_ratio_2023 = -0.0015  # -0.15%


Average Premium per Policy (2023): $911.32


---
### 🧮 Statewide Earthquake Insurance Indicators

This section provides statewide-level context for earthquake insurance across California. These variables are not merged into `gdf_ca_extended`, but they support interpretation of regional take-up rates and risk modeling.


#### 📉 Earthquake Insurance Loss Ratio (2023)

**Description:** Ratio of claims paid to premiums earned.  
**Source:** *California Department of Insurance — 2023 Market Share Report*

- **Loss Ratio:** –0.0015 (–0.15%)  
- EQ insurance continues to be a **low-claim business line**, likely due to high deductibles and rare payouts.


#### 💰 Average Earthquake Premium per Policy (2023)

**Description:** Statewide average residential earthquake premium.  
**Source:** *CDI — 2023 Residential Premium & Policy Summary*

- **Total EQ Premiums:** \$1,472,288,223  
- **Total EQ Policies:** 1,615,561  
- **Average Premium:** **$911.45** per policy

Higher average premiums may discourage adoption in lower-income or low-risk counties.

> ℹ️ These indicators provide macro-level insights into California’s earthquake insurance market. While not used directly in modeling, they strengthen context for reporting and interpretation.
